В этом практикуме мы рассмотрим работу с библиотекой **Gensim** для работы с векторными представлениями текста

Мы рассмотрим
- **Word2Vec** - векторные представления слов
- **FastText** - улучшенные представления с учетом морфологии  
- **Doc2Vec** - векторные представления документов


In [1]:
!pip install gensim

import gensim
import gensim.downloader as api
from gensim.models import Word2Vec, FastText, Doc2Vec
from gensim.models.doc2vec import TaggedDocument
import numpy as np

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 26.9 MB/s eta 0:00:00


## Часть 1: Word2Vec

### Что такое Word2Vec?

Word2Vec преобразует слова в векторы чисел так, что семантически похожие слова оказываются близко в векторном пространстве.

**Два основных алгоритма:**
- **CBOW** - предсказывает слово по контексту
- **Skip-gram** - предсказывает контекст по слову

**Загрузка предобученной модели**

In [2]:
w2v_model = api.load('glove-wiki-gigaword-100')

print(f"Размер словаря: {len(w2v_model.key_to_index)}")
print(f"Размерность векторов: {w2v_model.vector_size}")

[==================================================] 100.0% 128.1/128.1MB downloaded
Размер словаря: 400000
Размерность векторов: 100


Найдите документацию `gensim`: какие датасеты кроме `glove-wiki-gigaword-100` доступны в библиотеке?

Выберите 3 датасета и кратко опишите их (источник данных, примерный объем, зачем такой датасет может использоваться)

# 1. 20-newsgroups
The notorious collection of approximately 20,000 newsgroup posts, partitioned (nearly) evenly across 20 different newsgroups.

# 2. fake-news
News dataset, contains text and metadata from 244 websites and represents 12,999 posts in total from a specific window of 30 days. The data was pulled using the webhose.io API, and because it's coming from their crawler, not all websites identified by their BS Detector are present in this dataset. Data sources that were missing a label were simply assigned a label of 'bs'. There are (ostensibly) no genuine, reliable, or trustworthy news sources represented in this dataset (so far), so don't trust anything you read.

# 3. patent-2017
Patent Grant Full Text. Contains the full text including tables, sequence data and 'in-line' mathematical expressions of each patent grant issued in 2017.

**Базовые операции с векторами**

In [3]:
# Получаем вектор слова
vector = w2v_model['computer']
print(f"Вектор слова 'computer': {vector[:5]}...")  # Показываем первые 5 чисел

# Вычисляем схожесть между словами
similarity = w2v_model.similarity('computer', 'laptop')
print(f"Схожесть 'computer' и 'laptop': {similarity:.4f}")

Вектор слова 'computer': [-0.16298   0.30141   0.57978   0.066548  0.45835 ]...
Схожесть 'computer' и 'laptop': 0.7024


**Поиск похожих слов**

In [4]:
# Находим похожие слова
similar_words = w2v_model.most_similar('python', topn=5)
print("Слова, похожие на 'python':")
for word, score in similar_words:
    print(f"  {word}: {score:.4f}")

Слова, похожие на 'python':
  monty: 0.6886
  php: 0.5865
  perl: 0.5784
  cleese: 0.5447
  flipper: 0.5113


**Задание**

1. Загрузите любой датасет из gensim на ваш выбор

In [11]:
dataset = api.load("20-newsgroups")

2. Напишите функцию, которая принимает на вход любое слово и вовращает 10 наиболее близких по вектору слов

In [15]:
data = list(api.load("20-newsgroups"))

texts = [gensim.utils.simple_preprocess(doc['data']) for doc in data]
w2v_model = Word2Vec(sentences=texts, vector_size=100, window=5, min_count=5, workers=4)

def get_similar_words(word, topn=10):
    word = word.lower()
    if word in w2v_model.wv:
        return w2v_model.wv.most_similar(word, topn=topn)
    return f"Слово '{word}' не найдено в модели"

# Пример вызова
similar = get_similar_words('computer')
print(similar)

[('network', 0.6568306088447571), ('engineering', 0.5915011167526245), ('electrical', 0.5900196433067322), ('carnegie', 0.5803753137588501), ('computing', 0.5707860589027405), ('tech', 0.561071515083313), ('equipment', 0.5547621846199036), ('lab', 0.5499417781829834), ('rolm', 0.5459955930709839), ('manufacturing', 0.5381580591201782)]


3. Обучите модель Word2Vec на тестовом датасете из ячейки ниже

Примените следующие настройки:

- размер вектора: 50
- размер окна: 3
- минимальная частота слова: 1
- потоков: 2
- использовать skip-gram

In [16]:
cooking_sentences = [
    ['варить', 'суп', 'овощи', 'морковь', 'картофель'],
    ['жарить', 'курица', 'сковорода', 'масло', 'специи'],
    ['печь', 'хлеб', 'мука', 'дрожжи', 'духовка'],
    ['резать', 'овощи', 'салат', 'помидоры', 'огурцы'],
    ['смешивать', 'ингредиенты', 'тесто', 'яйца', 'молоко'],
    ['варить', 'паста', 'вода', 'соль', 'соус'],
    ['гриль', 'мясо', 'овощи', 'уголь', 'барбекю'],
    ['тушить', 'говядина', 'горшок', 'вино', 'травы'],
    ['запекать', 'рыба', 'лимон', 'духовка', 'фольга'],
    ['готовить', 'завтрак', 'яичница', 'бекон', 'тост'],
    ['месить', 'тесто', 'пирог', 'начинка', 'яблоки'],
    ['кипятить', 'вода', 'чай', 'кофе', 'чашка'],
    ['мариновать', 'мясо', 'соус', 'специи', 'холодильник'],
    ['взбивать', 'сливки', 'сахар', 'десерт', 'торт'],
    ['парить', 'овощи', 'здоровое', 'питание', 'брокколи']
]

In [28]:
model = Word2Vec(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2,
    sg=1
)

In [20]:
print(f"Слова в словаре: {list(model.wv.key_to_index.keys())[:10]}...")

Слова в словаре: ['овощи', 'мясо', 'соус', 'вода', 'тесто', 'духовка', 'специи', 'варить', 'брокколи', 'питание']...


4. Проверьте модель

In [21]:
# Проверяем похожие слова в кулинарной тематике
try:
    similar = model.wv.most_similar('варить', topn=5)
    print("Слова, похожие на 'варить':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'варить' не найдено в словаре")

Слова, похожие на 'варить':
  вино: 0.2398
  ингредиенты: 0.2172
  хлеб: 0.1938
  брокколи: 0.1846
  кипятить: 0.1711


In [22]:
# Найдите слова, похожие на "духовка"
word1 = 'духовка'
if word1 in cooking_model.wv:
    similar1 = cooking_model.wv.most_similar(word1, topn=5)
    print(f"Слова, похожие на '{word1}':")
    for similar_word, score in similar1:
        print(f"  {similar_word}: {score:.4f}")
else:
    print(f"Слово '{word1}' не найдено в модели")
print()

# Найдите слова, похожие на "овощи"
word2 = 'овощи'
if word2 in cooking_model.wv:
    similar2 = cooking_model.wv.most_similar(word2, topn=5)
    print(f"Слова, похожие на '{word2}':")
    for similar_word, score in similar2:
        print(f"  {similar_word}: {score:.4f}")
else:
    print(f"Слово '{word2}' не найдено в модели")

Слова, похожие на 'духовка':
  ингредиенты: 0.3199
  десерт: 0.3064
  холодильник: 0.2705
  питание: 0.2243
  пирог: 0.2142

Слова, похожие на 'овощи':
  мариновать: 0.2716
  хлеб: 0.2691
  гриль: 0.2546
  фольга: 0.2409
  сахар: 0.2108


## Часть 2: FastText

FastText улучшает Word2Vec, рассматривая слова как наборы символов (n-грамм). Это позволяет работать с редкими словами и опечатками

5. Обучите FastText на корпусе текстов из пункта 3. Используйте код ниже

In [24]:
ft_model = FastText(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2
)

6. Найдите слова, похожие на "варить", "духовка" и "овощи" с помощью обученной модели. Используйте код из пункта 4

In [25]:
word1 = 'варить'
if word1 in ft_model.wv:
    similar1 = ft_model.wv.most_similar(word1, topn=5)
    print(f"Слова, похожие на '{word1}':")
    for similar_word, score in similar1:
        print(f"  {similar_word}: {score:.4f}")
else:
    print(f"Слово '{word1}' не найдено в модели")
print()

word2 = 'духовка'
if word2 in ft_model.wv:
    similar2 = ft_model.wv.most_similar(word2, topn=5)
    print(f"Слова, похожие на '{word2}':")
    for similar_word, score in similar2:
        print(f"  {similar_word}: {score:.4f}")
else:
    print(f"Слово '{word2}' не найдено в модели")
print()

word3 = 'овощи'
if word3 in ft_model.wv:
    similar3 = ft_model.wv.most_similar(word3, topn=5)
    print(f"Слова, похожие на '{word3}':")
    for similar_word, score in similar3:
        print(f"  {similar_word}: {score:.4f}")
else:
    print(f"Слово '{word3}' не найдено в модели")

Слова, похожие на 'варить':
  жарить: 0.5353
  парить: 0.4805
  месить: 0.3541
  тушить: 0.3405
  специи: 0.2622

Слова, похожие на 'духовка':
  взбивать: 0.4565
  лимон: 0.3561
  салат: 0.3050
  курица: 0.3041
  тост: 0.2944

Слова, похожие на 'овощи':
  жарить: 0.2960
  фольга: 0.2574
  морковь: 0.2297
  соус: 0.2172
  торт: 0.2094


7. Сравните модели

Дана функция для сравнения Word2Vec и FastText

Придумайте 3 слова с опечатками и проверьте, найдет ли их FastText и Word2Vec

In [29]:
def compare_models(word):
    """Сравнивает представления слова в разных моделях"""
    print(f"\nСравнение для слова: '{word}'")

    # Word2Vec
    try:
        w2v_similar = model.wv.most_similar(word, topn=2)
        print(f"  Word2Vec: {[w for w, _ in w2v_similar]}")
    except KeyError:
        print(f"  Word2Vec: слово не найдено")

    # FastText
    try:
        ft_similar = ft_model.wv.most_similar(word, topn=2)
        print(f"  FastText: {[w for w, _ in ft_similar]}")
    except KeyError:
        print(f"  FastText: слово не найдено")

# Сравниваем для разных слов
compare_models('learning')
compare_models('neural')
compare_models('computr')
compare_models('lik')
compare_models('fust')


Сравнение для слова: 'learning'
  Word2Vec: слово не найдено
  FastText: ['духовка', 'пирог']

Сравнение для слова: 'neural'
  Word2Vec: слово не найдено
  FastText: ['мука', 'травы']

Сравнение для слова: 'computr'
  Word2Vec: слово не найдено
  FastText: ['взбивать', 'духовка']

Сравнение для слова: 'lik'
  Word2Vec: слово не найдено
  FastText: ['взбивать', 'запекать']

Сравнение для слова: 'fust'
  Word2Vec: слово не найдено
  FastText: ['чай', 'готовить']


## Часть 3: Doc2Vec

Doc2Vec расширяет Word2Vec для создания векторных представлений целых документов (предложений, абзацев, статей)

In [30]:
# Создаем размеченные документы
documents = [
    "machine learning is interesting",
    "deep learning uses neural networks",
    "python programming for data science",
    "artificial intelligence is amazing",
    "computer vision processes images"
]

# Преобразуем в формат TaggedDocument
tagged_docs = []
for i, doc in enumerate(documents):
    tokens = doc.split()
    tagged_doc = TaggedDocument(words=tokens, tags=[f"doc_{i}"])
    tagged_docs.append(tagged_doc)

print("Размеченные документы:")
for doc in tagged_docs[:3]:
    print(f"  Слова: {doc.words}")
    print(f"  Тег: {doc.tags}")

Размеченные документы:
  Слова: ['machine', 'learning', 'is', 'interesting']
  Тег: ['doc_0']
  Слова: ['deep', 'learning', 'uses', 'neural', 'networks']
  Тег: ['doc_1']
  Слова: ['python', 'programming', 'for', 'data', 'science']
  Тег: ['doc_2']


In [31]:
# Обучаем Doc2Vec
doc_model = Doc2Vec(
    documents=tagged_docs,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2,
    epochs=20
)

print("Doc2Vec модель обучена!")
print(f"Количество документов: {len(doc_model.dv.key_to_index)}")

Doc2Vec модель обучена!
Количество документов: 5


In [32]:
# Получаем вектор документа
doc_vector = doc_model.dv["doc_0"]
print(f"Вектор документа doc_0: {doc_vector[:5]}...")

# Находим похожие документы
similar_docs = doc_model.dv.most_similar("doc_0", topn=2)
print("\nДокументы, похожие на doc_0:")
for doc_tag, similarity in similar_docs:
    doc_id = int(doc_tag.split('_')[1])
    print(f"  {doc_tag}: {similarity:.4f}")
    print(f"    Текст: {documents[doc_id]}")

Вектор документа doc_0: [-0.01057    -0.01198188 -0.01982618  0.01710627  0.00710373]...

Документы, похожие на doc_0:
  doc_1: 0.2735
    Текст: deep learning uses neural networks
  doc_2: 0.1275
    Текст: python programming for data science


In [33]:
# Сравниваем схожесть документов
def compare_documents(doc1_id, doc2_id):
    similarity = doc_model.dv.similarity(f"doc_{doc1_id}", f"doc_{doc2_id}")
    print(f"Схожесть doc_{doc1_id} и doc_{doc2_id}: {similarity:.4f}")
    print(f"  doc_{doc1_id}: {documents[doc1_id]}")
    print(f"  doc_{doc2_id}: {documents[doc2_id]}")

compare_documents(0, 1)  # machine learning vs deep learning
compare_documents(0, 3)  # machine learning vs AI

Схожесть doc_0 и doc_1: 0.2735
  doc_0: machine learning is interesting
  doc_1: deep learning uses neural networks
Схожесть doc_0 и doc_3: -0.0822
  doc_0: machine learning is interesting
  doc_3: artificial intelligence is amazing


8. Сравните схожесть doc_2 и doc_4

In [34]:
similarity_2_4 = doc_model.dv.similarity("doc_2", "doc_4")
print(f"Схожесть doc_2 и doc_4: {similarity_2_4:.4f}")
print(f"  doc_2: {documents[2]}")
print(f"  doc_4: {documents[4]}")

Схожесть doc_2 и doc_4: -0.0362
  doc_2: python programming for data science
  doc_4: computer vision processes images


9. Найдите самый похожий документ на doc_1

In [35]:
similar_docs = doc_model.dv.most_similar("doc_1", topn=1)
print(f"Самый похожий документ на doc_1:")
for doc_tag, similarity in similar_docs:
    doc_id = int(doc_tag.split('_')[1])
    print(f"  {doc_tag}: {similarity:.4f}")
    print(f"    Текст: {documents[doc_id]}")

Самый похожий документ на doc_1:
  doc_0: 0.2735
    Текст: machine learning is interesting


10. Выберите любую из трёх моделей. Обучите модели с разной размерностью (10, 50, 100). Продемонстрируйте качество их работы на примере поиска похожих слов (выберите любые 3 примера, соответствующих тематике корпуса из пункта 4)

In [36]:
def train_and_test(vector_size):
    print(f"\n{'='*50}")
    print(f"МОДЕЛЬ С РАЗМЕРНОСТЬЮ ВЕКТОРОВ: {vector_size}")
    print(f"{'='*50}")

    model = Word2Vec(
        sentences=cooking_sentences,
        vector_size=vector_size,
        window=3,
        min_count=1,
        workers=2,
        sg=1
    )

    test_words = ['овощи', 'жарить', 'духовка']

    for word in test_words:
        if word in model.wv:
            similar = model.wv.most_similar(word, topn=3)
            print(f"\nСлова, похожие на '{word}':")
            for similar_word, score in similar:
                print(f"  {similar_word}: {score:.4f}")
        else:
            print(f"Слово '{word}' не найдено в модели")

    return model

model_10 = train_and_test(10)
model_50 = train_and_test(50)
model_100 = train_and_test(100)


МОДЕЛЬ С РАЗМЕРНОСТЬЮ ВЕКТОРОВ: 10

Слова, похожие на 'овощи':
  жарить: 0.7185
  фольга: 0.7146
  горшок: 0.6696

Слова, похожие на 'жарить':
  овощи: 0.7185
  мясо: 0.6996
  горшок: 0.6287

Слова, похожие на 'духовка':
  взбивать: 0.5916
  тушить: 0.5880
  говядина: 0.5443

МОДЕЛЬ С РАЗМЕРНОСТЬЮ ВЕКТОРОВ: 50

Слова, похожие на 'овощи':
  мариновать: 0.2716
  хлеб: 0.2691
  гриль: 0.2546

Слова, похожие на 'жарить':
  парить: 0.3108
  взбивать: 0.3092
  торт: 0.2703

Слова, похожие на 'духовка':
  ингредиенты: 0.3199
  десерт: 0.3064
  холодильник: 0.2705

МОДЕЛЬ С РАЗМЕРНОСТЬЮ ВЕКТОРОВ: 100

Слова, похожие на 'овощи':
  взбивать: 0.2190
  питание: 0.2163
  бекон: 0.1956

Слова, похожие на 'жарить':
  смешивать: 0.2497
  огурцы: 0.2161
  духовка: 0.1904

Слова, похожие на 'духовка':
  жарить: 0.1904
  парить: 0.1670
  завтрак: 0.1627
